# 1. Read the Dataset

In [0]:
import pandas as pd 
from matplotlib import pyplot as plt 
import seaborn as sns 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import FunctionTransformer
from mlflow.models import infer_signature
import mlflow
import mlflow.sklearn
import numpy as np

## Read and Convert to Parquet

### a) Read data

In [0]:
df_train = spark.read.csv(
    "/Volumes/workspace/default/data_customers/customer_churn_dataset-training-master.csv",
    header=True,
    inferSchema=True,
)

df_test = spark.read.csv(
    "/Volumes/workspace/default/data_customers/customer_churn_dataset-testing-master.csv",
    header=True,
    inferSchema=True,
)

### b) Convert To Parquet

#### Training Set

In [0]:
# Lưu tập Train
df_train.write.mode("overwrite").parquet("/Volumes/workspace/default/data_customers/parquet/train")

In [0]:
df_train = spark.read.parquet(
    "/Volumes/workspace/default/data_customers/parquet/train"
)

In [0]:
df_train.printSchema()


In [0]:
display(df_train.show(5))

##### Testing Set

In [0]:
# Lưu tập Test
df_test.write.mode("overwrite").parquet("/Volumes/workspace/default/data_customers/parquet/test")

In [0]:
df_test = spark.read.parquet(
    "/Volumes/workspace/default/data_customers/parquet/test"
)

In [0]:
df_test.printSchema()

In [0]:
df_test.show(5)

In [0]:
#Read file by using AWS
# df_train = spark.read.csv(
#     "s3a://churn-databricks/raw/customer_churn_dataset-training-master.csv",
#     header=True,
#     inferSchema=True
# )

# df_test = spark.read.csv(
#     "s3a://churn-databricks/raw/customer_churn_dataset-testing-master.csv",
#     header=True,
#     inferSchema=True
# )

In [0]:
# df = (
#     spark.readStream.format("cloudFiles")
#         .option("cloudFiles.format", "csv")  # Chọn format nguồn
#         .option("header", "true")
#         .option("inferSchema", "true")
#         .load("s3a://churn-databricks/raw/")  # Thư mục S3 gốc
# )

# # Ghi sang  Parquet)
# (df.writeStream
#     .format("parquet")  
#     .option("checkpointLocation", "/mnt/checkpoints/churn")
#     .outputMode("append")
#     .start("/mnt/delta/churn")  # Đường dẫn cho Delta table
# )

In [0]:
# %python
# df_train = spark.read.csv(
#     "/Volumes/workspace/default/data_customers/customer_churn_dataset-training-master.csv",
#     header=True,
#     inferSchema=True,
# )

# df_test = spark.read.csv(
#     "/Volumes/workspace/default/data_customers/customer_churn_dataset-testing-master.csv",
#     header=True,
#     inferSchema=True,
# )


# 2. EDA

In [0]:
df_train = df_train.drop("CustomerID")
df_test = df_test.drop("CustomerID")

In [0]:
%python
df_train.printSchema()

In [0]:
df_test.printSchema()

In [0]:
%python
display(df_train.describe())

In [0]:
%python
from pyspark.sql.functions import col, sum, when

missing_df_train= df_train.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_train.columns
])

display(missing_df_train)

In [0]:
%python
df_train = df_train.dropna()
df_test = df_test.dropna()


df_train.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df_train.columns
]).show()


In [0]:
df_train = df_train.dropDuplicates()
df_test = df_test.dropDuplicates()

In [0]:
%python
for c in df_train.columns:
    print(c, df_train.select(c).distinct().count())

In [0]:
%python
import matplotlib.pyplot as plt
import seaborn as sns

churn_pd = df_train.select("Churn").toPandas()

plt.figure(figsize=(6,4))
ax = sns.countplot(x='Churn', data=churn_pd)

for p in ax.patches:
    ax.annotate(
        f'{int(p.get_height())}',
        (p.get_x() + p.get_width()/2., p.get_height()),
        ha='center',
        va='bottom'
    )

plt.show()

In [0]:
num_cols = [
    f.name for f in df_train.schema.fields
    if f.dataType.simpleString() in ["int", "double"]
    and f.name != "Churn"
]

print("Numeric columns:", num_cols)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

for col in num_cols:
    
    # convert cột Spark -> Pandas
    data = df_train.select(col).toPandas()[col]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Histogram
    axes[0].hist(data, bins=30, density=True, alpha=0.7)
    axes[0].set_title(f"Histogram of {col}")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Density")
    axes[0].grid(alpha=0.3)

    # KDE
    sns.kdeplot(data, fill=True, ax=axes[1], color='orange')
    axes[1].set_title(f"KDE of {col}")
    axes[1].set_xlabel(col)
    axes[1].set_ylabel("Density")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Nếu data lớn thì sample trước
sample_df = df_train.sample(fraction=0.2, seed=42)

for col in num_cols:
    
    # Spark -> Pandas
    data = sample_df.select(col).toPandas()[col]

    fig, axes = plt.subplots(1, 2, figsize=(12,4))

    # Boxplot
    axes[0].boxplot(
        data,
        vert=False,
        patch_artist=True,
        boxprops=dict(facecolor='skyblue', alpha=0.7),
        medianprops=dict(color='red')
    )
    axes[0].set_title(f"Boxplot of {col}")
    axes[0].set_xlabel(col)
    axes[0].grid(axis='x', alpha=0.3)

    # Violin Plot
    sns.violinplot(
        x=data,
        ax=axes[1],
        inner='quartile',
        color='orange'
    )
    axes[1].set_title(f"Violin Plot of {col}")
    axes[1].set_xlabel(col)
    axes[1].grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

In [0]:
from pyspark.sql.functions import avg

churn_rate = (
    df_train
    .groupBy("Support Calls")
    .agg(avg("Churn").alias("churn_rate"))
    .orderBy("Support Calls")
)

churn_pd = churn_rate.toPandas()

plt.figure(figsize=(8,5))
plt.bar(churn_pd['Support Calls'], churn_pd['churn_rate'])

plt.xlabel("Number of Support Calls")
plt.ylabel("Churn Rate")
plt.title("Churn Rate by Support Calls")
plt.ylim(0,1)

plt.show()

In [0]:
cols = num_cols + ["Churn"]
pdf = df_train.select(cols).toPandas()

corr = pdf.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title("Correlation Matrix including Churn")
plt.show()

### Check Categorical Columns

In [0]:
categorical_cols = [f.name for f in df_train.schema.fields 
                    if f.dataType.simpleString() == 'string']
categorical_cols

In [0]:
for col in categorical_cols:
    print(f"\n===== {col} =====")
    df_train.groupBy(col).count().orderBy("count", ascending=False).show()

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert toàn bộ categorical columns sang pandas một lần
pdf = df_train.select(categorical_cols).toPandas()

for col in categorical_cols:
    plt.figure(figsize=(6,4))
    
    sns.countplot(
        x=pdf[col],
        order=pdf[col].value_counts().index
    )
    
    plt.title(f"{col} Distribution")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

### Data Engineering : New Column

In [0]:
from pyspark.sql.functions import when, col

# Train
df_train = df_train.withColumn(
    "Age_group",
    when(col("Age") < 30, "Young (<30)")
    .when((col("Age") >= 30) & (col("Age") < 50), "Adult (30-50)")
    .otherwise("Senior (50+)")
)

# Test
df_test = df_test.withColumn(
    "Age_group",
    when(col("Age") < 30, "Young (<30)")
    .when((col("Age") >= 30) & (col("Age") < 50), "Adult (30-50)")
    .otherwise("Senior (50+)")
)

In [0]:
import matplotlib.pyplot as plt

# Spark -> Pandas
pdf = df_train.select("Age_group", "Churn").toPandas()

# Group & pivot bằng pandas
pivot = (
    pdf.groupby(["Age_group", "Churn"])
       .size()
       .unstack(fill_value=0)
)

# Plot
plt.figure(figsize=(12,6))
pivot.plot(kind='bar')
plt.title('Age Group vs Churn')
plt.xlabel('Age Group')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Churn')
plt.tight_layout()
plt.show()

### Check Last Interaction with Churn

In [0]:
import matplotlib.pyplot as plt

# Spark -> Pandas
pdf = df_train.select("Last Interaction", "Churn").toPandas()

# Group + pivot
pivot = (
    pdf.groupby(["Last Interaction", "Churn"])
       .size()
       .unstack(fill_value=0)
)

# Chuẩn hoá theo % từng hàng
pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

# Plot
plt.figure(figsize=(12,6))
pivot_norm.plot(kind='bar', stacked=True)

plt.title("Percentage Distribution of Churn by Last Interaction")
plt.ylabel("Percentage")
plt.xlabel("Last Interaction")
plt.legend(title="Churn")
plt.tight_layout()
plt.show()

# 4.Feature Engineering

In [0]:
from pyspark.sql.functions import col

# ===== TRAIN =====
df_train = df_train \
    .withColumn("Usage_Per_Tenure",
                col("Usage Frequency") / (col("Tenure") + 1)) \
    .withColumn("Spend_Per_Usage",
                col("Total Spend") / (col("Usage Frequency") + 1)) \
    .withColumn("Spend_Per_Tenure",
                col("Total Spend") / (col("Tenure") + 1)) \
    .withColumn("Payment_Delay_Ratio",
                col("Payment Delay") / 30) \
    .withColumn("Engagement_Score",
                (col("Usage Frequency") * col("Total Spend")) / 1000)

# ===== TEST =====
df_test = df_test \
    .withColumn("Usage_Per_Tenure",
                col("Usage Frequency") / (col("Tenure") + 1)) \
    .withColumn("Spend_Per_Usage",
                col("Total Spend") / (col("Usage Frequency") + 1)) \
    .withColumn("Spend_Per_Tenure",
                col("Total Spend") / (col("Tenure") + 1)) \
    .withColumn("Payment_Delay_Ratio",
                col("Payment Delay") / 30) \
    .withColumn("Engagement_Score",
                (col("Usage Frequency") * col("Total Spend")) / 1000)

### Modeling

In [0]:
# Convert to pandas
df_train = df_train.toPandas()
df_test = df_test.toPandas()

# Split data train
X_train = df_train.drop(columns=['Churn']) 
y_train = df_train['Churn']

In [0]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

In [0]:
# X_train, X_val, y_train, y_val = train_test_split(
#     X, y, 
#     test_size=0.2, 
#     random_state=42, 
#     stratify=y
# )

X_test = df_test.drop(columns=['Churn'])
y_test = df_test['Churn']

In [0]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [0]:
raw_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=0.1, penalty='l2', class_weight='balanced', random_state=42),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_split=10, class_weight='balanced', random_state=42), 
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, class_weight='balanced', random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, subsample=0.8, random_state=42)
}


results = {}

for name, model in raw_models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    cv_results = cross_validate(pipeline, X_train, y_train, cv=5, scoring=['accuracy', 'recall', 'roc_auc'])
    
    results[name] = {
        'AUC': cv_results['test_roc_auc'].mean(),
        'Recall': cv_results['test_recall'].mean(),
        'Accuracy': cv_results['test_accuracy'].mean()
    }

In [0]:
results_df = pd.DataFrame(results).T.sort_values(by='AUC', ascending=False)
print(results_df)

In [0]:
best_model_name = results_df.index[0]
final_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', raw_models[best_model_name])])

final_pipeline.fit(X_train, y_train)

y_test_pred = final_pipeline.predict(X_test)
print(classification_report(y_test, y_test_pred))

### Testing data

In [0]:
# Khai báo signature với dữ liệu huấn luyện và đầu ra dự đoán
signature = infer_signature(X_train, final_pipeline.predict(X_train))

In [0]:
def feature_engineering(df):
    df = df.copy()

    # Tránh chia cho 0 bằng cách dùng clip
    tenure_safe = df["Tenure"].clip(lower=1)
    usage_safe = df["Usage Frequency"].clip(lower=1)

    # ===== Engineered Features =====
    df["Usage_Per_Tenure"] = df["Usage Frequency"] / tenure_safe
    df["Spend_Per_Usage"] = df["Total Spend"] / usage_safe
    df["Spend_Per_Tenure"] = df["Total Spend"] / tenure_safe
    df["Payment_Delay_Ratio"] = df["Payment Delay"] / 30   # giống Spark
    df["Engagement_Score"] = (
        df["Usage Frequency"] * df["Total Spend"]
    ) / 1000

    # ===== Age Group (giống Spark logic hơn) =====
    df["Age_group"] = np.where(
        df["Age"] < 30, "Young (<30)",
        np.where(
            df["Age"] < 50, "Adult (30-50)",
            "Senior (50+)"
        )
    )

    return df


full_pipeline = Pipeline([
    ("feature_engineering", FunctionTransformer(feature_engineering)),
    ("preprocessor", preprocessor),
    ("classifier", raw_models[best_model_name])
])

In [0]:
new_customer = pd.DataFrame([{
    "Age": 45,                  # Khách hàng trưởng thành, ổn định
    "Gender": "Female",
    "Tenure": 60,               # Thâm niên 60 tháng (5 năm - rất lâu)
    "Usage Frequency": 30,      # Dùng dịch vụ liên tục (30 ngày/tháng)
    "Support Calls": 0,         # KHÔNG BAO GIỜ gọi phàn nàn hay báo lỗi
    "Payment Delay": 0,         # Luôn thanh toán đúng hạn (Trễ = 0)
    "Subscription Type": "Premium",
    "Contract Length": "12 months", 
    "Total Spend": 3500.0,      # Tổng chi tiêu cực kỳ cao
    "Last Interaction": 1       # Vừa mới dùng dịch vụ ngày hôm qua
}])

new_customer = feature_engineering(new_customer)

prediction = full_pipeline.predict(new_customer)[0]
probability = full_pipeline.predict_proba(new_customer)[0][1]

print("Prediction:", prediction)
print("Probability:", round(probability, 4))


### Save model

In [0]:
#Save model
with mlflow.start_run(run_name=best_model_name):

    # Log parameters
    mlflow.log_param("model_name", best_model_name)

    # Log metrics
    mlflow.log_metric("accuracy", 0.56)  # bạn có thể tính lại
    mlflow.log_metric("auc", results_df.loc[best_model_name, "AUC"])
    mlflow.log_metric("recall", results_df.loc[best_model_name, "Recall"])

    # Log model
    mlflow.sklearn.log_model(final_pipeline, "model", signature=signature)

print("Model logged to MLflow!")